# 01 · Classical Baseline Segmenter — Component 3

**Phase 1, Step 1.1.** Runs `segmentation/classical.py` (illumination flattening → cell-centre
detection → marker watershed) on **train-split images only** and checks it against Konan.

| # | Section | Output (in `outputs/classical/`) |
|---|---------|----------------------------------|
| 1 | Example overlays (full image + zoom) | `01_examples.png` |
| 2 | Batch run on train images | `02_batch_results.csv` |
| 3 | Comparison with Konan (mean area, ECD, CV) | `03_vs_konan.png` |
| 4 | Parameter sanity: cell-size sensitivity | `04_sensitivity.png` |

ECD here uses the **preliminary** scale (0.752 µm/px) — a sanity check, not a reported result.
A CPU runtime is enough.

## 0 · Setup

In [ ]:
# Mount Drive, pull the latest code FIRST, then run the shared setup script.
from google.colab import drive; drive.mount("/content/drive")
REPO_DIR = "/content/drive/MyDrive/Colab Notebooks/Research_Project/xai-cornea-grading"
!git -C "{REPO_DIR}" pull --ff-only
%run "{REPO_DIR}/ml/cellular/colab_setup.py"

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from scipy import stats
from tqdm.auto import tqdm

from config import load_config
from segmentation.classical import ClassicalParams, segment, overlay

cfg = load_config()
P = cfg["paths"]
OUT = P["outputs_dir"] / "classical"
OUT.mkdir(parents=True, exist_ok=True)
SEED = cfg["split"]["seed"]
UM_PER_PX = cfg["scale"]["um_per_px_prelim"]
params = ClassicalParams.from_config(cfg)
sns.set_theme(style="whitegrid", context="notebook")

man = pd.read_csv(P["splits_dir"] / "manifest.csv")
train = man[man["split"] == "train"].copy()          # never touch val / test here
print(f"Train images: {len(train)}  |  params: {params}")
print(f"Expected hexagon area: {params.expected_area_px:.0f} px")


def load_gray(image_file):
    return cv2.imread(str(P["images_dir"] / image_file), cv2.IMREAD_GRAYSCALE)


def save(fig, name):
    fig.savefig(OUT / f"{name}.png", dpi=150, bbox_inches="tight")
    plt.show()

## 1 · Example overlays
Green = kept complete cells, red = watershed walls, darkened = area too dark to analyse.
Check: do red lines sit on the dark cell walls? Are big cells split or small cells merged?

In [ ]:
examples = train.groupby("ecd_band").sample(n=2, random_state=SEED)   # 2 per ECD band

fig, axes = plt.subplots(len(examples), 2, figsize=(16, 5.2 * len(examples)))
for row_axes, (_, r) in zip(axes, examples.iterrows()):
    img = load_gray(r["image_file"])
    res = segment(img, params)
    ov = overlay(img, res)

    # zoom: 300x300 window centred on the densest part of the kept cells
    ys, xs = np.nonzero(res.labels)
    cy, cx = (int(np.median(ys)), int(np.median(xs))) if len(ys) else (img.shape[0] // 2, img.shape[1] // 2)
    y0 = int(np.clip(cy - 150, 0, img.shape[0] - 300)); x0 = int(np.clip(cx - 150, 0, img.shape[1] - 300))

    row_axes[0].imshow(ov); row_axes[0].add_patch(plt.Rectangle((x0, y0), 300, 300, fill=False, ec="yellow", lw=2))
    row_axes[0].set_title(f"{r['image_id']}  |  {res.n_cells} cells  |  Konan ECD {r['ecd']:.0f}, "
                          f"ours {res.ecd(UM_PER_PX):.0f}", fontsize=10)
    row_axes[1].imshow(ov[y0:y0 + 300, x0:x0 + 300]); row_axes[1].set_title("zoom (yellow box)", fontsize=10)
    for ax in row_axes:
        ax.axis("off")
save(fig, "01_examples")

## 2 · Batch run on train images
Runs on a random sample of train images (default 150, ≈ 1–2 min). Per image: number of complete
cells, mean area, CV, and ECD with the preliminary scale.

In [ ]:
N_BATCH = 150
batch = train.sample(min(N_BATCH, len(train)), random_state=SEED)

rows = []
for _, r in tqdm(batch.iterrows(), total=len(batch), desc="Segmenting"):
    res = segment(load_gray(r["image_file"]), params)
    rows.append({
        "image_id": r["image_id"],
        "n_cells": res.n_cells,
        "mean_area_px": res.mean_area_px,
        "mean_area_um2": res.mean_area_px * UM_PER_PX ** 2,
        "cv": res.cv_percent(),
        "ecd": res.ecd(UM_PER_PX),
        "valid_fraction": float(res.valid.mean()),
        "konan_num_cells": r["num_cells"], "konan_mean_area": r["mean_area"],
        "konan_cv": r["cv"], "konan_ecd": r["ecd"],
    })
results = pd.DataFrame(rows)
results.to_csv(OUT / "02_batch_results.csv", index=False)
display(results.describe().T.round(1))
print(f"Cells per image: median {results['n_cells'].median():.0f} "
      f"(Konan median {results['konan_num_cells'].median():.0f})")

## 3 · Comparison with Konan
* **Mean area / ECD**: shows whether walls are found at the right spacing. A constant ratio means a
  scale offset (fixable in Phase 4); a large spread means segmentation errors.
* **CV**: sensitive to over-/under-segmentation. Konan measures ~110 hand-picked cells, we measure
  every complete cell in the valid area, so some difference is expected.

In [ ]:
def pct_err(a, b):
    return (100 * (a - b).abs() / b)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (ours, konan, label) in zip(axes, [("mean_area_um2", "konan_mean_area", "Mean cell area (µm²)"),
                                           ("ecd", "konan_ecd", "ECD (cells/mm²)"),
                                           ("cv", "konan_cv", "CV (%)")]):
    d = results[[ours, konan]].dropna()
    ax.scatter(d[konan], d[ours], s=10, alpha=0.6)
    lims = [min(d.min()), max(d.max())]
    ax.plot(lims, lims, "r--", lw=1, label="y = x")
    rho, _ = stats.spearmanr(d[konan], d[ours])
    ax.set(xlabel=f"Konan {label}", ylabel=f"Classical {label}",
           title=f"{label}\nSpearman ρ = {rho:.2f}, median |err| = {pct_err(d[ours], d[konan]).median():.1f}%")
    ax.legend()
save(fig, "03_vs_konan")

summary = {
    "n_images": len(results),
    "median_cells_per_image": float(results["n_cells"].median()),
    "area_ratio_median (ours/konan)": float((results["mean_area_um2"] / results["konan_mean_area"]).median()),
    "ecd_median_abs_pct_err": float(pct_err(results["ecd"], results["konan_ecd"]).median()),
    "cv_median_abs_pct_err": float(pct_err(results["cv"], results["konan_cv"]).median()),
}
print(pd.Series(summary).round(3).to_string())

## 4 · Sensitivity to the cell-size assumption
The segmenter scales everything by `cell_spacing_px`. If results change a lot when it is ±15 %,
the method is fragile; ideally the mean area changes much less than the parameter.

In [ ]:
from dataclasses import replace

sample = batch.head(30)
factors = [0.85, 1.0, 1.15]
sens = []
for f in factors:
    p = replace(params, cell_px=params.cell_px * f)
    for _, r in sample.iterrows():
        res = segment(load_gray(r["image_file"]), p)
        sens.append({"factor": f, "image_id": r["image_id"], "mean_area_px": res.mean_area_px,
                     "n_cells": res.n_cells})
sens = pd.DataFrame(sens)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(data=sens, x="factor", y="mean_area_px", ax=axes[0]); axes[0].set_title("Mean cell area vs cell_px factor")
sns.boxplot(data=sens, x="factor", y="n_cells", ax=axes[1]); axes[1].set_title("Cells per image vs cell_px factor")
save(fig, "04_sensitivity")
display(sens.groupby("factor")[["mean_area_px", "n_cells"]].median().round(1))

## Notes (fill in after running)
* Visual quality of walls (Section 1): …
* Agreement with Konan (Section 3): …
* Sensitivity (Section 4): …
* Typical failure modes (to fix during annotation): …